<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Fine-Tune Computer Vision Models

---

### About
Fine-tuning is a method of modifying generalized CVs to have better performance in specific domains.

### Learning Objective
- Fine tune an image model

### Notebook Guide
- Setup the Model
   - System Setup
   - Setup your model
   - Create a custom dataset class
   - Setup data
- Fine Tune the model!
    - Training Setup
    - Training Loop
    - Monitoring the training metrics
    - Save the Model
    - Test the Model on New Images
- Try It!

## Installs

In [ ]:
# Install these as needed

# !pip install torch torchvision
# !pip install transformers
# !pip install matplotlib
# !pip install pycocotools

## Imports

In [ ]:


from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
import time
import random

In [ ]:
# Load an image from the data set to inspect it

healthy_image = Image.open("../data/grape-plants/Grape_healthy/0cd6cdbd-1674-4abb-b734-756da4994cd0___Mt.N.V_HL 6052_180deg.JPG")
healthy_image

In [ ]:
# Load an unhealthy image for comparison

rot_image= Image.open("../data/grape-plants/Grape_Black_rot/0a06c482-c94a-44d8-a895-be6fe17b8c06___FAM_B.Rot 5019_flipLR.JPG")
rot_image

## System Setup 
Check your system setup. GPU is required for real world fine tuning. This demo can be run on CPU. Note it takes about 10 minutes to run the training step on a standard personal computer with CPU.

In [ ]:
# Check you compute system


## Setup your model

Load a pre-trained model. Here we are using a `ViT` which is on the smaller side (will run faster) but is still effective. 

[Model Card](https://huggingface.co/google/vit-base-patch16-224-in21k)


Note: For CPU-only training, we're using a smaller model (`ViT-tiny`) to speed up training.

In [ ]:
# Use a smaller model for CPU training


## Create a custom dataset class

- You will process images on-the-fly rather than loading everything into memory at once, which is more memory efficient. 
  - On-the-fly processing is when the dataset class handles preprocessing each image at access time rather than preprocessing everything upfront, spreading the computational load throughout training. 
- It will also allow you to efficiently pull training and test data as well as data labels. 

In [ ]:
class GrapeLeafDataset(Dataset):


## Setup data 
1. Collect the labels from the image paths
2. Create training and validation datasets

In [ ]:
# Define classes and data paths


# Set seed for reproducibility


In [ ]:
# Collect image paths and labels

    # Get all image files in the directory

    # Limit to 50 images per class for faster training


In [ ]:
# Create train/validation split (80/20)


In [ ]:
# Create data sets and dataloaders


# Smaller batch size for CPU training



# Fine Tune the model! 
## Training Setup

Define optimizer, loss function, and training loop

In our `train_epoch()` function below we will be fine tuning our base model (ViT here).  

- `loss.backward()`  # Calculates gradients
- `optimizer.step()` # Makes small adjustments to model's parameters based on the new data

The entire training loop (calling `train_epoch` repeatedly for multiple epochs) is where the fine-tuning magic happens. With each epoch, the model parameters are incrementally adjusted to better classify our specific grape disease images.

##### What's happening behind the scenes:

- The model is slowly adapting its understanding of image features from general objects to specific grape disease patterns.
- Early layers of the network (which detect basic features) change less.
- Later layers (which combine features into classifications) change more significantly.

In [ ]:
# Define training parameters


# Set up optimizer and loss function


In [ ]:
# Define training and evaluation functions
def train_epoch():
 

        # Move data to device
        
        # Zero the gradients
        
        # Forward pass
        
        # Calculate loss
        
        # Backward pass and optimize
        
        # Track stats
        
        # Print progress every 5 batches

    
    # Calculate epoch metrics


def evaluate():

            
            # Forward pass

            
            # Calculate loss

            
            # Get predictions

    
    # Calculate metrics


## Training Loop

In [ ]:
# Note: this can take almost 11 minutes to run - time for coffee!

# Track metrics

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

# Start training

    
    # Train for one epoch

    
    # Evaluate on validation set

    
    # Save metrics


## Monitoring the training metrics 
It is important to monitor the training and validation metrics - they show you how effectively the fine-tuning process is working as the model adapts from its general pre-training to your specific task.

_Be on the look out for the following:_

### Positive Patterns

- **Decreasing Loss**: Both training and validation loss should generally decrease over time. This indicates the model is learning useful patterns from the data.
- **Improving Accuracy**: Both training and validation accuracy should increase over time, showing that the model is getting better at classifying the images correctly.
- **Convergence**: Eventually, the metrics should start to plateau, indicating the model has learned what it can from the data.
- **Close Performance**: Ideally, training and validation metrics should be relatively close to each other, suggesting the model generalizes well.

### Warning Signs

- **Overfitting**: If training loss/accuracy continues to improve while validation metrics worsen or plateau, the model is likely memorizing the training data rather than learning generalizable patterns. This is visible as a widening gap between training and validation curves.
- **Underfitting**: If both training and validation metrics remain poor and show minimal improvement, the model may be too simple for the task, or the learning rate might be too low.
- **Erratic Validation Metrics**: If validation metrics jump around significantly between epochs, it could indicate that the validation set is too small or not representative.
- **Too-Perfect Training**: If training accuracy quickly reaches 100% while validation lags behind, this is a clear sign of overfitting.

### Note for the demo
Due to our small validation set (to support CPU processing) our accuracy is high (1.0) immediately. In the real world this would indicate that we may not have representative data or may have a data leakage issue. For this example we will ignore it. 

In [ ]:
# Plot loss and accuracy curves
plt.figure(figsize=(12, 5))

# Plot loss
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Training')
plt.plot(val_losses, label='Validation')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Plot accuracy
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label='Training')
plt.plot(val_accuracies, label='Validation')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## Save the model
The model will save in the `02-cv-fine-tuning` folder.

In [ ]:
# Save the model


## Test the model on new images

In [ ]:
def predict_image(image_path, model, feature_extractor, device):
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    inputs = feature_extractor(images=image, return_tensors="pt")
    pixel_values = inputs['pixel_values'].to(device)
    
    # Get prediction
    model.eval()
    with torch.no_grad():
        outputs = model(pixel_values=pixel_values)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        prediction = torch.argmax(logits, dim=1).item()
    
    # Get class name and probability
    class_name = classes[prediction]
    probability = probabilities[0][prediction].item()
    
    return class_name, probability, image

In [ ]:
# Test on sample images
test_images = [
    # Add paths to test images here
    # For example:
    "../data/grape-plants/Grape_healthy/0cd6cdbd-1674-4abb-b734-756da4994cd0___Mt.N.V_HL 6052_180deg.JPG",
    "../data/grape-plants/Grape_Black_rot/0a06c482-c94a-44d8-a895-be6fe17b8c06___FAM_B.Rot 5019_flipLR.JPG"
]

In [ ]:
# plot predictions


# Try It! 
Select new images and test the model.

In [ ]:
# Test on sample images and plot predictions
